# Retail Sales Prediction
Individual regression capstone. Run with the VS Code Jupyter extension.

**Data required:** Rossmann train.csv and store.csv in data/raw. No results are fabricated.

Student: add your name and roll number before submission.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'retail').exists():
    raise RuntimeError('Open the project root as your VS Code workspace.')
print(sys.executable)
MAX_STORES = None  # Full data first; use 100 only if resources require it
from retail.core import load_data, split_dates, features
data, stores = load_data(ROOT / 'data/raw', MAX_STORES)
data.head()

## Day 1: Data quality
The target is daily store sales. A future forecast must use information available before that day. Preserve real zero sales. Missing labels and duplicate store dates require investigation rather than silent deletion.

In [ ]:
import pandas as pd
display(data.dtypes.to_frame('dtype'))
display(data.isna().sum().to_frame('missing'))
print('Duplicate store dates:', data.duplicated(['Store', 'Date']).sum())
print('Rows:', len(data), 'Stores:', data.Store.nunique())
print('Range:', data.Date.min(), data.Date.max())
train, validation, test = split_dates(data)
for label, part in [('train',train),('validation',validation),('test',test)]:
    print(label, part.Date.min(), part.Date.max(), len(part))

## Exploratory analysis
Use training dates only so test patterns do not guide modeling. The trend averages observed stores and may reflect changing store coverage. Boxplots hide outlier points for readability; they retain observations. Promotion comparisons show association, not a causal effect.

In [ ]:
from retail.eda import create_charts
from IPython.display import Image, display
for chart in create_charts(ROOT / 'data/raw', ROOT / 'artifacts/figures', MAX_STORES):
    display(Image(filename=str(chart)))

### Your checkpoint
Write at least one specific observation for each plot. Which weekdays have higher typical sales? Is the promotion comparison confounded by store or weekday? Why might store types differ? Record observations here after executing the plots.

## Day 2: Features and models
Ridge fits an additive linear relationship with an L2 penalty. Random Forest averages decision trees and captures interactions. A store/weekday mean is the reference baseline. Categorical store IDs do not imply an ordered scale. Numeric missingness is imputed inside each fitted pipeline. We exclude Customers because its same-day value is unknown at forecast time.

The final 42 days are held out for test and the preceding 42 for validation. We fit on open days and apply zero for known closures. Validation MAE chooses the final method, including the baseline if it wins.

In [ ]:
X = features(train)
assert 'Customers' not in X and 'Sales' not in X
assert train.Date.max() < validation.Date.min()
assert validation.Date.max() < test.Date.min()
display(X.head())

In [ ]:
from retail.train import train as run_training
metrics = run_training(ROOT / 'data/raw', ROOT / 'artifacts', MAX_STORES)
rows = [{'model':name, 'subset':subset, **values} for name,result in metrics['validation'].items() for subset,values in result.items()]
display(pd.DataFrame(rows))
print('Selected:', metrics['selected_model'])
display(pd.DataFrame(metrics['test']).T)

### Result review
MAE measures average absolute sales error. RMSE emphasizes larger misses. R² compares squared error with a constant mean reference and can be negative. Compare open-day metrics because predictable closures can make aggregate performance look stronger. Do not tune after inspecting test scores.

Explain: Did the selected model beat the baseline on validation? How does open-day error differ? What practical inventory or staffing decision could this forecast inform?

In [ ]:
import matplotlib.pyplot as plt
predictions = pd.read_csv(ROOT / 'artifacts/test_predictions.csv', parse_dates=['Date'])
ax = predictions.groupby('Date')[['Sales','Prediction']].mean().plot(figsize=(10,4), title='Test period: observed versus predicted mean daily sales')
ax.set_ylabel('Sales (dataset units)')
plt.tight_layout()
plt.show()

## Day 3: Consistent inference
The app uses the same saved preprocessing/model pipeline and store lookup. Dates must follow the final fitting date. Forecasts far beyond the tested six weeks have unverified reliability.

In [ ]:
import joblib
from retail.core import predict_one
bundle = joblib.load(ROOT / 'artifacts/model.joblib')
store = bundle['known_stores'][0]
forecast_date = pd.Timestamp(bundle['training_end']) + pd.Timedelta(days=1)
print('Open:', predict_one(bundle,store,forecast_date,1,1,'0',0))
assert predict_one(bundle,store,forecast_date,0,0,'0',0) == 0

Run `.venv\Scripts\python.exe -m streamlit run app.py` in the terminal. See docs/DEPLOYMENT.md for publishing.

**Viva:** Why save preprocessing? Why reject an unknown store? Why does a forest struggle to extrapolate a trend? Why does a promotion flag need to be known in advance?

## Day 4: Report and presentation
Run `python scripts/build_report.py` after training. Regenerate the deck using scripts/build_slides.mjs with the documented artifact runtime, or edit the provided deck with your measured results. Add your own plot interpretations, student details and verified deployment URL.

**Final checkpoint:** Restart kernel and Run All. Explain leakage, chronological splitting, model selection, MAE versus RMSE, limitations, and future rolling-origin validation.